In [1]:
import phonlp
from triplet_extraction.src.triplet_extraction import init_vncorenlp, load_synonym_dict, load_stopwords
import os

current_dir = os.getcwd()
base_dir = os.path.dirname(current_dir)
print(f"Working directory: {current_dir}")
print(f"Base directory set to: {base_dir}\n")

# === Define files paths relative to base directory ===
vncorenlp_dir = os.path.join(current_dir, "nlp_models", "VnCoreNLP-1.2")
phonlp_dir = os.path.join(current_dir, "nlp_models", "phonlp")
synonym_file = os.path.join(current_dir, "listSameKey.txt")
stopwords_file = os.path.join(current_dir, "stopwords.csv")
no_triplet_csv_path = os.path.join(current_dir, "logs", "no_triplets_dat_dai_log_1.csv")
log_file_path = os.path.join(current_dir, "logs", "dat_dai_triplet_extraction.txt")

# === Initialize NLP models ===
vncorenlp_client = init_vncorenlp(vncorenlp_dir)
phoNLP_model = phonlp.load(save_dir=phonlp_dir)
synonym_dict = load_synonym_dict(synonym_file)
stopwords = load_stopwords(stopwords_file)

Working directory: E:\Github\LawAssistant\triplet_extraction
Base directory set to: E:\Github\LawAssistant

Loading model from: E:\Github\LawAssistant\triplet_extraction\nlp_models\phonlp/phonlp.pt


In [1]:
from triplet_extraction.src.db import init_mongo
# === Initialize MongoDB ===
mongo_client = init_mongo()
db = mongo_client["KB_PROPERTY_LAW"]
documents_col = db["documents"]
sections_col = db["legal_sections"]
process_sections_col = db["processed_legal_sections"]

You successfully connected to MongoDB!


In [2]:
from triplet_extraction.src.db import extract_all_from_mongo_collection

for doc in extract_all_from_mongo_collection(process_sections_col):
    print(doc)
    break

{'_id': ObjectId('695fce3297fdc5f42bd4a6f0'), 'section_id': ObjectId('694fb816aedc69db48c709a6'), 'sequence': 1, 'content': 'Tổ chức trong nước là cơ quan nhà nước.', 'so_hieu': '31/2024/QH15'}


In [24]:
sections = sections_col.find(
    {
        "is_amendment": True,
        "type": {"$in": ["điểm"]},
    }
)
for sec in sections:
    print(sec['_id'])

694fb816aedc69db48c71062
694fb816aedc69db48c71114
694fb816aedc69db48c7134f
694fb816aedc69db48c71350
694fb816aedc69db48c71351
694fb816aedc69db48c71365
694fb816aedc69db48c71366
694fb816aedc69db48c71368
694fb816aedc69db48c71369
694fb816aedc69db48c7136b
694fb816aedc69db48c7136c
694fb816aedc69db48c71372
694fb816aedc69db48c71373
694fb816aedc69db48c71523
694fb816aedc69db48c71829
694fb816aedc69db48c719ba
694fb817aedc69db48c71e2c
694fb817aedc69db48c71e2d
694fb817aedc69db48c71e63
694fb817aedc69db48c71f2e
694fb817aedc69db48c71f5b
694fb817aedc69db48c71f84
694fb817aedc69db48c720d2
694fb817aedc69db48c72323
694fb817aedc69db48c72324
694fb817aedc69db48c7232a
694fb817aedc69db48c7232c
694fb817aedc69db48c7232e
694fb817aedc69db48c72333
694fb817aedc69db48c72334
694fb817aedc69db48c72335
694fb817aedc69db48c72336
694fb817aedc69db48c72339
694fb817aedc69db48c7233a
694fb817aedc69db48c7233b
694fb817aedc69db48c7233c
694fb817aedc69db48c72347
694fb817aedc69db48c72348
694fb817aedc69db48c72349
694fb817aedc69db48c7234c


In [20]:
import re
from pymongo import UpdateOne

AMENDMENT_PATTERN = re.compile(
    r"^\s*(sửa đổi|bổ sung|bãi bỏ|thay thế)\b",
    flags=re.IGNORECASE
)

bulk_ops = []
BATCH_SIZE = 1000

total_modified = 0
total_scanned = 0

cursor = sections_col.find(
    {"type": {"$in": ["điểm", "khoản", "điều"]}},
)

for sec in cursor:
    total_scanned += 1

    content = sec.get("content")
    old_value = sec.get("is_amendment")
    new_value = bool(AMENDMENT_PATTERN.search(content))

    if old_value != new_value:
        bulk_ops.append(
            UpdateOne(
                {"_id": sec["_id"]},
                {"$set": {"is_amendment": new_value}}
            )
        )

    if len(bulk_ops) >= BATCH_SIZE:
        result = sections_col.bulk_write(bulk_ops, ordered=False)
        total_modified += result.modified_count
        bulk_ops.clear()

# Flush remaining
if bulk_ops:
    result = sections_col.bulk_write(bulk_ops, ordered=False)
    total_modified += result.modified_count

print(f"Scanned  : {total_scanned}")
print(f"Updated  : {total_modified}")
print("Done rechecking is_amendment.")

Scanned  : 27314
Updated  : 246
Done rechecking is_amendment.


In [6]:
content = "Đăng ký danh sách định giá viên và việc thay đổi, bổ sung danh sách định giá viên với cơ quan có chức năng quản lý đất đai cấp tỉnh nơi đăng ký trụ sở chính;"
new_value = bool(AMENDMENT_PATTERN.search(content))
print(new_value)

False


In [12]:
from bson import ObjectId

SECTION_ID = "694fb81aaedc69db48c74462"

sections = process_sections_col.find(
    {"section_id": ObjectId(SECTION_ID)}
)
sections = list(sections)
for sec in sections:
    print(sec)

{'_id': ObjectId('695fbb4f97fdc5f42bd4581f'), 'sequence': 1, 'section_id': ObjectId('694fb81aaedc69db48c74462'), 'content': 'Hành vi chuyển đất rừng đặc dụng sang loại đất khác trong nhóm đất nông nghiệp bị xử phạt.', 'so_hieu': '123/2024/NĐ-CP'}
{'_id': ObjectId('695fbb4f97fdc5f42bd45820'), 'section_id': ObjectId('694fb81aaedc69db48c74462'), 'sequence': 2, 'content': 'Hành vi chuyển đất rừng phòng hộ sang loại đất khác trong nhóm đất nông nghiệp bị xử phạt.', 'so_hieu': '123/2024/NĐ-CP'}
{'_id': ObjectId('695fbb4f97fdc5f42bd45821'), 'sequence': 3, 'section_id': ObjectId('694fb81aaedc69db48c74462'), 'content': 'Hành vi chuyển đất rừng sản xuất sang loại đất khác trong nhóm đất nông nghiệp bị xử phạt.', 'so_hieu': '123/2024/NĐ-CP'}
{'_id': ObjectId('695fbb4f97fdc5f42bd45822'), 'section_id': ObjectId('694fb81aaedc69db48c74462'), 'sequence': 4, 'content': 'Hình thức và mức xử phạt đối với hành vi chuyển đất rừng đặc dụng, đất rừng phòng hộ, đất rừng sản xuất sang loại đất khác trong nhóm 

In [ ]:
logs_file = r"E:\Github\LawAssistant\triplet_extraction\logs\dat_dai_triplet_extraction.txt"

with open("example.txt", "r", encoding="utf-8") as f:
    for line in f:
        print(line.strip())

In [13]:
from triplet_extraction.src.triplet_extraction import clean_text, parsing_result


def parse_dataframe_to_tokens(df):
    """Convert DataFrame to a list of token dicts"""
    tokens = []
    for _, row in df.iterrows():
        token = {
            'id': int(row['id']),
            'word': str(row['word']),
            'pos': str(row['pos']),
            'head': int(row['head']),
            'deprel': str(row['deprel'])
        }
        tokens.append(token)
    return tokens


def split_sentence_np_vp(tokens):
    if not tokens:
        return [], []

    root_index = -1
    root_id = None

    # Find the main verb (root or first valid verb)
    for i, token in enumerate(tokens):
        if token['pos'] == 'V':
            if token['deprel'] == 'root' and token['head'] == 0:
                # Avoid picking verb at start (index 0)
                if i == 0:
                    continue
                root_index = i
                root_id = token['id']
                break
            elif root_index == -1 and token['deprel'] != 'nmod':
                # Avoid first word if it's a verb
                if i == 0:
                    continue
                root_index = i
                root_id = token['id']

    # Fallback – pick next verb if root not found
    if root_index == -1:
        for i, token in enumerate(tokens):
            if token['pos'] == 'V' and i > 0:  # skip first position
                root_index = i
                root_id = token['id']
                break

    # Final split
    if root_index != -1 and root_id is not None:
        np_tokens = tokens[:root_index]
        vp_tokens = tokens[root_index:]
        return np_tokens, vp_tokens

    return [], []

def collect_dependents(tokens, head_id):
    """Return set of token ids: head_id + all recursive dependents"""
    subtree = {head_id}
    result = []
    added = True
    while added:
        added = False
        for token in tokens:
            if token['head'] in subtree and token['id'] not in subtree:
                subtree.add(token['id'])
                result.append(token)
                added = True
    return result


def collect_direct_dependents(tokens, head_id):
    """Return list of token dicts that directly depend on head_id"""
    return [t for t in tokens if t['head'] == head_id]


def rebuild_phrase(tokens):
    """Sort tokens by their original position in the sentence and join them together"""
    tokens_sorted = sorted(tokens, key=lambda x: x['id'])
    phrase = " ".join(t['word'] for t in tokens_sorted)
    return phrase

def extract_main_subjects(np_tokens):
    if not np_tokens:
        return []

    sub_tokens = [t for t in np_tokens if t['deprel'] == 'sub']
    if not sub_tokens:
        sub_tokens = [t for t in np_tokens if t['deprel'] == 'root']
    if not sub_tokens:
        return []

    main_subjects = [sub_tokens[0]]
    main_subjects.extend(collect_direct_dependents(np_tokens, sub_tokens[0]['id']))
    if main_subjects and main_subjects[-1]['pos'] in ['Cc', 'CH']:
        main_subjects.pop()
    if len(main_subjects) == len(np_tokens):
        return [rebuild_phrase(np_tokens)]

    # Find Coordination Word (Cc, CH)
    coord_tokens = [t for t in np_tokens if (t['pos'] in ['Cc', 'CH'] and t not in main_subjects)]
    non_main_tokens = set()
    if len(coord_tokens) > 0:
        phrases = []

        for coord in coord_tokens:
            coord_index = next((i for i, t in enumerate(np_tokens) if t['id'] == coord['id']), None)

            left_tokens = []
            main_subjects_id = [obj['id'] for obj in main_subjects]
            for i in range(coord_index - 1, -1, -1):
                token = np_tokens[i]
                if token['pos'] not in ['CH', 'Cc'] and token['id'] not in main_subjects_id:
                    left_tokens.append(token)
                    non_main_tokens.add(token['id'])
                else:
                    break

            right_tokens = []
            for i in range(coord_index + 1, len(np_tokens)):
                token = np_tokens[i]
                if token['pos'] not in ['CH', 'Cc']:
                    right_tokens.append(token)
                    non_main_tokens.add(token['id'])
                else:
                    break

            if left_tokens:
                phrases.append(rebuild_phrase(left_tokens))
            if right_tokens:
                phrases.append(rebuild_phrase(right_tokens))

        # Remove duplicates while preserving order
        phrases = list(dict.fromkeys(phrases))

        for sub in main_subjects:
            if sub['id'] in non_main_tokens:
                main_subjects.remove(sub)
        main_subject_phrase = rebuild_phrase(main_subjects)

        # Properly combine main subject with each phrase
        combined_phrases = []
        for phrase in phrases:
            combined_phrases.append(main_subject_phrase + " " + phrase)

        if not combined_phrases:
            return [main_subject_phrase]

        return combined_phrases
    else:
        return [rebuild_phrase(np_tokens)]

def extract_verbs(vp_tokens):
    if not vp_tokens:
        return [], []

    # Find the root verb first
    root_verb = None
    for t in vp_tokens:
        if t['deprel'] == 'root' and t['head'] == 0 and t['pos'] == 'V':
            root_verb = t
            break

    # Fallback to the first verb that not nmod or aux
    if not root_verb:
        for t in vp_tokens:
            if t['pos'] == 'V' and t['deprel'] not in ['nmod', 'aux']:
                root_verb = t
                break

    # Fallback to any first verb in vp_tokens
    if not root_verb:
        for t in vp_tokens:
            if t['pos'] == 'V':
                root_verb = t
                break

    if not root_verb:
        return [vp_tokens[0]['word']], [vp_tokens[0]]

    # Check for coordination markers (CH, Cc) that are direct dependents of root
    coord_markers = [t for t in vp_tokens if t['pos'] in ['Cc', 'CH'] and t['head'] == root_verb['id']]

    if coord_markers:
        coordinated_verbs = [root_verb]
        for t in vp_tokens:
            if t['pos'] == 'V' and t['head'] == root_verb['id'] and t['deprel'] in ['vmod', 'conj']:
                coordinated_verbs.append(t)

        # Sort by ID to maintain order
        coordinated_verbs.sort(key=lambda x: x['id'])

        verb_phrases = []
        all_tokens = []
        coordinated_verbs_id = [v['id'] for v in coordinated_verbs]
        for verb in coordinated_verbs:
            phrase_tokens = [verb]
            dependents = collect_direct_dependents(vp_tokens, verb['id'])

            # Keep only dependents that are not other coordinated verbs or coordination markers
            dependents = [d for d in dependents if
                          d['id'] not in coordinated_verbs_id
                          and d['pos'] not in ['CH', 'Cc']
                          and d['deprel'] == 'vmod']

            phrase_tokens.extend(dependents)
            all_tokens.extend(phrase_tokens)

            verb_phrases.append({
                'text': rebuild_phrase(phrase_tokens),
                'tokens': phrase_tokens
            })

        return verb_phrases, all_tokens

    # Single verb: return it with its dependents
    verb_tokens = [root_verb]
    verb_tokens.extend(collect_direct_dependents(vp_tokens, root_verb['id']))
    sorted_verb_tokens = sorted(verb_tokens, key=lambda x: x['id'])

    # Filter out tokens after the first noun
    filtered_tokens = []
    for token in sorted_verb_tokens:
        if token['pos'].startswith('N'):
            break
        filtered_tokens.append(token)

    # If we filtered out everything, at least return the root verb
    if not filtered_tokens:
        filtered_tokens = [root_verb]

    return [{
        'text': rebuild_phrase(filtered_tokens),
        'tokens': filtered_tokens
    }], filtered_tokens

def extract_objects(vp_tokens, verb_token):
    # Remove verb tokens from vp_tokens (make a copy to avoid modifying during iteration)
    vp_tokens = [t for t in vp_tokens if t['id'] not in [v['id'] for v in verb_token]]

    if not vp_tokens:
        return []

    # Find the first object token (dob, iob, pob)
    obj_token = next((t for t in vp_tokens if t['deprel'] in ['dob', 'iob', 'pob']), None)

    # Fallback to the first noun in vp_tokens
    if obj_token is None:
        obj_token = next((t for t in vp_tokens if t['pos'] == 'N'), None)

    if obj_token is None:
        obj_token = next((t for t in vp_tokens if t['deprel'] == 'vmod'), None)

    if obj_token is None:
        return []

    # Collect main object and its dependents
    main_objects = [obj_token]
    main_objects.extend(collect_direct_dependents(vp_tokens, obj_token['id']))

    if len(main_objects) == len(vp_tokens):
        return [{
            'text': rebuild_phrase(vp_tokens),
            'tokens': vp_tokens
        }]

    # Find coordination tokens
    coord_tokens = [t for t in vp_tokens if t['pos'] in ['Cc', 'CH']]
    for obj in main_objects:
        if obj in coord_tokens:
            main_objects = []
            break

    if coord_tokens:
        combined_phrases = []

        for coord in coord_tokens:
            coord_index = next((i for i, t in enumerate(vp_tokens) if t['id'] == coord['id']), None)

            # LEFT TOKENS
            left_tokens = []
            for i in range(coord_index - 1, -1, -1):
                token = vp_tokens[i]
                if token['pos'] not in ['CH', 'Cc'] and token['id'] not in [obj['id'] for obj in main_objects]:
                    left_tokens.append(token)
                else:
                    break
            left_tokens = left_tokens[::-1]

            # RIGHT TOKENS
            right_tokens = []
            for i in range(coord_index + 1, len(vp_tokens)):
                token = vp_tokens[i]
                if token['pos'] not in ['CH', 'Cc']:
                    right_tokens.append(token)
                else:
                    break

            # Combine main object with left and right tokens
            for tokens_side in [left_tokens, right_tokens]:
                if tokens_side:
                    combined_phrases.append({
                        'text': rebuild_phrase(main_objects) + " " + rebuild_phrase(tokens_side),
                        'tokens': main_objects + tokens_side
                    })

        # Remove duplicates while preserving order
        seen = set()
        final_phrases = []
        for item in combined_phrases:
            if item['text'] not in seen:
                final_phrases.append(item)
                seen.add(item['text'])

        return final_phrases

    else:
        return [{
            'text': rebuild_phrase(vp_tokens),
            'tokens': vp_tokens
        }]


def process_sentence(df, logger):
    tokens = parse_dataframe_to_tokens(df)
    np_tokens, vp_tokens = split_sentence_np_vp(tokens)
    logger.debug("-----------------NP-----------------")
    logger.debug(np_tokens)
    logger.debug("-----------------VP-----------------")
    logger.debug(vp_tokens)

    # Extract subjects, verbs, objects
    subjects = extract_main_subjects(np_tokens)
    verbs, verbs_token = extract_verbs(vp_tokens)
    objects = extract_objects(vp_tokens, verbs_token)

    logger.debug("-----------------subjects----------------")
    logger.debug(subjects)
    logger.debug("-----------------verbs----------------")
    for verb in verbs:
        logger.debug(verb['text'])
    logger.debug("-----------------objects----------------")
    for obj in objects:
        logger.debug(obj['text'])

    verbs_position = {}
    for verb in verbs:
        verb_last_id = verb['tokens'][0]['id']
        verbs_position[verb['text']] = verb_last_id

    verbs_sorted = sorted(verbs, key=lambda v: v['tokens'][0]['id'])
    objects_sorted = sorted(objects, key=lambda o: o['tokens'][0]['id'])

    # Start combine them into triplets
    triplets = []
    for subj in subjects:
        for i, verb in enumerate(verbs_sorted):
            verb_last_id = verb['tokens'][-1]['id']

            # Determine the next verb's first ID (or infinity if this is the last verb)
            next_verb_first_id = verbs_sorted[i + 1]['tokens'][0]['id'] if i + 1 < len(verbs_sorted) else float('inf')

            # Objects that come after this verb but before the next verb
            obj_candidates = []
            for obj in objects_sorted:
                obj_id = obj['tokens'][-1]['id']
                if verb_last_id < obj_id < next_verb_first_id:
                    obj_candidates.append(obj)

            for obj in obj_candidates:
                triplets.append((subj, verb['text'], obj['text']))

    return triplets

def triplet_extraction(text, vncorenlp_client, phoNLP_model, stopwords, logger, max_depth=2, depth=0):
    """Recursively extract triplets from text, including nested subjects/objects"""
    if depth > max_depth or not text.strip():
        return []

    sentence = clean_text(text)
    segmented_text = vncorenlp_client.word_segment(sentence)

    # Annotate text
    annotation = phoNLP_model.annotate(text=segmented_text[0])
    df = parsing_result(annotation)
    print(df.to_string(index=False))

    triplets = process_sentence(df, logger)
    all_triplets = []

    for subj, verb, obj in triplets:
        # Refine subject
        try:
            subj_annotation = phoNLP_model.annotate(text=subj)
            df_subj = parsing_result(subj_annotation)
            refined_subj_triplets = process_sentence(df_subj, logger)
            if refined_subj_triplets and len(refined_subj_triplets) > 0 and len(refined_subj_triplets[0]) > 0:
                subj_refined = refined_subj_triplets[0][0]
            else:
                subj_refined = subj
        except (IndexError, Exception):
            subj_refined = subj
            refined_subj_triplets = []

        # Refine object
        try:
            obj_annotation = phoNLP_model.annotate(text=obj)
            df_obj = parsing_result(obj_annotation)
            refined_obj_triplets = process_sentence(df_obj, logger)
            if refined_obj_triplets and len(refined_obj_triplets) > 0 and len(refined_obj_triplets[0]) > 0:
                obj_refined = refined_obj_triplets[0][0]
            else:
                obj_refined = obj
        except (IndexError, Exception):
            obj_refined = obj
            refined_obj_triplets = []

        all_triplets.append((subj_refined, verb, obj_refined))
        all_triplets.extend(refined_subj_triplets)
        all_triplets.extend(refined_obj_triplets)

    filtered_triplets = []

    for triplet in all_triplets:
        subj, verb, obj = triplet

        # Remove stopwords
        subj_filtered = ' '.join([w for w in subj.split() if w.lower() not in stopwords]).strip()
        verb_filtered = ' '.join([w for w in verb.split() if w.lower() not in stopwords]).strip()
        obj_filtered = ' '.join([w for w in obj.split() if w.lower() not in stopwords]).strip()

        # Skip remove stopwords if any element becomes empty
        if not subj_filtered:
            subj_filtered = subj
        if not verb_filtered:
            verb_filtered = verb
        if not obj_filtered:
            obj_filtered = obj

        subj_filtered = subj_filtered.replace('_', ' ').strip().lower()
        verb_filtered = verb_filtered.replace('_', ' ').strip().lower()
        obj_filtered = obj_filtered.replace('_', ' ').strip().lower()
        filtered_triplets.append((subj_filtered, verb_filtered, obj_filtered))

    return filtered_triplets



import logging
from triplet_extraction.src.triplet_extraction import setup_logger

os.makedirs(os.path.dirname(log_file_path), exist_ok=True)
logger, console_handler, file_handler = setup_logger(
    name="triplet_extraction",
    level=logging.DEBUG,
    log_to_file=False,
    file_path=log_file_path
)

# Disable console logging (optional)
logger.removeHandler(file_handler)

for sec in sections:
    print(sec["sequence"], sec["content"])
    triplets = triplet_extraction(
                    text=sec["content"],
                    vncorenlp_client=vncorenlp_client,
                    phoNLP_model=phoNLP_model,
                    stopwords=stopwords,
                    logger=logger,
                    max_depth=4,
                )

    print(f"Extracted {len(triplets)} triplets:")
for t in triplets:
    print(t)

1 Hành vi chuyển đất rừng đặc dụng sang loại đất khác trong nhóm đất nông nghiệp bị xử phạt.


100%|██████████| 1/1 [00:00<00:00,  3.10it/s]
[DEBUG] -----------------NP-----------------
[DEBUG] [{'id': 1, 'word': 'hành_vi', 'pos': 'N', 'head': 13, 'deprel': 'sub'}, {'id': 2, 'word': 'chuyển', 'pos': 'V', 'head': 1, 'deprel': 'nmod'}, {'id': 3, 'word': 'đất', 'pos': 'N', 'head': 2, 'deprel': 'dob'}, {'id': 4, 'word': 'rừng_đặc_dụng', 'pos': 'N', 'head': 3, 'deprel': 'nmod'}, {'id': 5, 'word': 'sang', 'pos': 'V', 'head': 2, 'deprel': 'vmod'}, {'id': 6, 'word': 'loại', 'pos': 'N', 'head': 5, 'deprel': 'dob'}, {'id': 7, 'word': 'đất', 'pos': 'N', 'head': 6, 'deprel': 'nmod'}, {'id': 8, 'word': 'khác', 'pos': 'A', 'head': 6, 'deprel': 'nmod'}, {'id': 9, 'word': 'trong', 'pos': 'E', 'head': 6, 'deprel': 'loc'}, {'id': 10, 'word': 'nhóm', 'pos': 'N', 'head': 9, 'deprel': 'pob'}, {'id': 11, 'word': 'đất', 'pos': 'N', 'head': 10, 'deprel': 'nmod'}, {'id': 12, 'word': 'nông_nghiệp', 'pos': 'N', 'head': 11, 'deprel': 'nmod'}]
[DEBUG] -----------------VP-----------------
[DEBUG] [{'id': 13,

 id          word pos head deprel
  1       hành_vi   N   13    sub
  2        chuyển   V    1   nmod
  3           đất   N    2    dob
  4 rừng_đặc_dụng   N    3   nmod
  5          sang   V    2   vmod
  6          loại   N    5    dob
  7           đất   N    6   nmod
  8          khác   A    6   nmod
  9         trong   E    6    loc
 10          nhóm   N    9    pob
 11           đất   N   10   nmod
 12   nông_nghiệp   N   11   nmod
 13            bị   V    0   root
 14       xử_phạt   V   13   vmod
 15             .  CH   13  punct
Extracted 0 triplets:
2 Hành vi chuyển đất rừng phòng hộ sang loại đất khác trong nhóm đất nông nghiệp bị xử phạt.


100%|██████████| 1/1 [00:00<00:00, 10.38it/s]
[DEBUG] -----------------NP-----------------
[DEBUG] [{'id': 1, 'word': 'hành_vi', 'pos': 'N', 'head': 13, 'deprel': 'sub'}, {'id': 2, 'word': 'chuyển', 'pos': 'V', 'head': 1, 'deprel': 'nmod'}, {'id': 3, 'word': 'đất', 'pos': 'N', 'head': 2, 'deprel': 'dob'}, {'id': 4, 'word': 'rừng_phòng_hộ', 'pos': 'N', 'head': 3, 'deprel': 'nmod'}, {'id': 5, 'word': 'sang', 'pos': 'V', 'head': 2, 'deprel': 'vmod'}, {'id': 6, 'word': 'loại', 'pos': 'N', 'head': 5, 'deprel': 'dob'}, {'id': 7, 'word': 'đất', 'pos': 'N', 'head': 6, 'deprel': 'nmod'}, {'id': 8, 'word': 'khác', 'pos': 'A', 'head': 6, 'deprel': 'nmod'}, {'id': 9, 'word': 'trong', 'pos': 'E', 'head': 6, 'deprel': 'loc'}, {'id': 10, 'word': 'nhóm', 'pos': 'N', 'head': 9, 'deprel': 'pob'}, {'id': 11, 'word': 'đất', 'pos': 'N', 'head': 10, 'deprel': 'nmod'}, {'id': 12, 'word': 'nông_nghiệp', 'pos': 'N', 'head': 11, 'deprel': 'nmod'}]
[DEBUG] -----------------VP-----------------
[DEBUG] [{'id': 13,

 id          word pos head deprel
  1       hành_vi   N   13    sub
  2        chuyển   V    1   nmod
  3           đất   N    2    dob
  4 rừng_phòng_hộ   N    3   nmod
  5          sang   V    2   vmod
  6          loại   N    5    dob
  7           đất   N    6   nmod
  8          khác   A    6   nmod
  9         trong   E    6    loc
 10          nhóm   N    9    pob
 11           đất   N   10   nmod
 12   nông_nghiệp   N   11   nmod
 13            bị   V    0   root
 14       xử_phạt   V   13   vmod
 15             .  CH   13  punct
Extracted 0 triplets:
3 Hành vi chuyển đất rừng sản xuất sang loại đất khác trong nhóm đất nông nghiệp bị xử phạt.


...

 id          word pos head deprel
  1       hành_vi   N   13    sub
  2        chuyển   V    1   nmod
  3           đất   N    2    dob
  4 rừng_sản_xuất   N    3   nmod
  5          sang   V    2   vmod
  6          loại   N    5    dob
  7           đất   N    6   nmod
  8          khác   A    6   nmod
  9         trong   E    6    loc
 10          nhóm   N    9    pob
 11           đất   N   10   nmod
 12   nông_nghiệp   N   11   nmod
 13            bị   V    0   root
 14       xử_phạt   V   13   vmod
 15             .  CH   13  punct
Extracted 0 triplets:
4 Hình thức và mức xử phạt đối với hành vi chuyển đất rừng đặc dụng, đất rừng phòng hộ, đất rừng sản xuất sang loại đất khác trong nhóm đất nông nghiệp là phạt tiền từ 2.000.000 đồng đến 3.000.000 đồng đối với diện tích đất dưới 0,5 héc ta.


100%|██████████| 1/1 [00:00<00:00,  6.85it/s]
[DEBUG] -----------------NP-----------------
[DEBUG] [{'id': 1, 'word': 'hình_thức', 'pos': 'N', 'head': 24, 'deprel': 'sub'}, {'id': 2, 'word': 'và', 'pos': 'Cc', 'head': 1, 'deprel': 'coord'}, {'id': 3, 'word': 'mức', 'pos': 'N', 'head': 2, 'deprel': 'conj'}, {'id': 4, 'word': 'xử_phạt', 'pos': 'V', 'head': 1, 'deprel': 'nmod'}, {'id': 5, 'word': 'đối_với', 'pos': 'E', 'head': 1, 'deprel': 'nmod'}, {'id': 6, 'word': 'hành_vi', 'pos': 'N', 'head': 5, 'deprel': 'pob'}, {'id': 7, 'word': 'chuyển', 'pos': 'V', 'head': 6, 'deprel': 'nmod'}, {'id': 8, 'word': 'đất', 'pos': 'N', 'head': 7, 'deprel': 'dob'}, {'id': 9, 'word': 'rừng_đặc_dụng', 'pos': 'N', 'head': 8, 'deprel': 'nmod'}, {'id': 10, 'word': ',', 'pos': 'CH', 'head': 8, 'deprel': 'punct'}, {'id': 11, 'word': 'đất', 'pos': 'N', 'head': 8, 'deprel': 'nmod'}, {'id': 12, 'word': 'rừng_phòng_hộ', 'pos': 'N', 'head': 11, 'deprel': 'nmod'}, {'id': 13, 'word': ',', 'pos': 'CH', 'head': 8, 'dep

 id          word pos head deprel
  1     hình_thức   N   24    sub
  2            và  Cc    1  coord
  3           mức   N    2   conj
  4       xử_phạt   V    1   nmod
  5       đối_với   E    1   nmod
  6       hành_vi   N    5    pob
  7        chuyển   V    6   nmod
  8           đất   N    7    dob
  9 rừng_đặc_dụng   N    8   nmod
 10             ,  CH    8  punct
 11           đất   N    8   nmod
 12 rừng_phòng_hộ   N   11   nmod
 13             ,  CH    8  punct
 14           đất   N    8   nmod
 15 rừng_sản_xuất   N   14   nmod
 16          sang   V    8   nmod
 17          loại   N   16    dob
 18           đất   N   17   nmod
 19          khác   A   17   nmod
 20         trong   E   17    loc
 21          nhóm   N   20    pob
 22           đất   N   21   nmod
 23   nông_nghiệp   N   22   nmod
 24            là   V    0   root
 25          phạt   V   24   vmod
 26          tiền   N   25    dob
 27            từ   E   25   vmod
 28     2.000.000   M   29    det
 29          đ

100%|██████████| 1/1 [00:00<00:00, 13.10it/s]
[DEBUG] -----------------NP-----------------
[DEBUG] [{'id': 1, 'word': 'hình_thức', 'pos': 'N', 'head': 0, 'deprel': 'root'}, {'id': 2, 'word': 'và', 'pos': 'Cc', 'head': 1, 'deprel': 'coord'}]
[DEBUG] -----------------VP-----------------
[DEBUG] [{'id': 3, 'word': 'xử_phạt', 'pos': 'V', 'head': 2, 'deprel': 'conj'}, {'id': 4, 'word': 'đối_với', 'pos': 'E', 'head': 1, 'deprel': 'vmod'}, {'id': 5, 'word': 'hành_vi', 'pos': 'N', 'head': 4, 'deprel': 'pob'}, {'id': 6, 'word': 'chuyển', 'pos': 'V', 'head': 5, 'deprel': 'nmod'}, {'id': 7, 'word': 'đất', 'pos': 'N', 'head': 6, 'deprel': 'dob'}, {'id': 8, 'word': 'rừng_đặc_dụng', 'pos': 'N', 'head': 7, 'deprel': 'nmod'}]
[DEBUG] -----------------subjects----------------
[DEBUG] ['hình_thức']
[DEBUG] -----------------verbs----------------
[DEBUG] xử_phạt
[DEBUG] -----------------objects----------------
[DEBUG] đối_với hành_vi chuyển đất rừng_đặc_dụng
100%|██████████| 1/1 [00:00<00:00, 12.82it/s]
[

Extracted 6 triplets:
('hình thức', 'phạt từ đối với', 'tiền 2.000.000 đồng đến 3.000.000 đồng diện tích đất 0,5 héc ta')
('hình thức', 'xử phạt', 'đối với hành vi chuyển đất rừng đặc dụng')
('hình thức', 'phạt từ đối với', 'tiền 2.000.000 đồng đến 3.000.000 đồng diện tích đất 0,5 héc ta')
('hình thức', 'xử phạt', 'đối với đất rừng phòng hộ')
('hình thức', 'phạt từ đối với', 'tiền 2.000.000 đồng đến 3.000.000 đồng diện tích đất 0,5 héc ta')
('hình thức', 'xử phạt', 'đối với đất rừng sản xuất sang đất trong nhóm đất nông nghiệp')


In [11]:
error_csv_file = r"E:\Github\LawAssistant\triplet_extraction\logs\no_triplets_dat_dai_log_1.csv"

import pandas as pd
error_df = pd.read_csv(error_csv_file)

unique_document_number = set()
for _, row in error_df.iterrows():
    unique_document_number.add(row['document_number'])

print(len(unique_document_number))
for doc_number in unique_document_number:
    print(doc_number)

51
103/2024/NĐ-CP
96/2024/NĐ-CP
09/2025/TT-BXD
58/2024/QH15
05/2024/TT-BXD
108/2024/NĐ-CP
46/2025/TT-BTC
91/2015/QH13
29/2024/TT-BTNMT
104/2024/NĐ-CP
10/2024/TT-BTNMT
21/2017/QH14
44/2022/NĐ-CP
140/2025/NĐ-CP
50/2024/TT-BTNMT
230/2025/NĐ-CP
04/2024/TT-BXD
03/2025/NĐ-CP
95/2024/NĐ-CP
151/2025/NĐ-CP
226/2025/NĐ-CP
11/2024/TT-BNV
131/2025/NĐ-CP
178/2025/NĐ-CP
88/2024/NĐ-CP
90/2025/QH15
101/2024/NĐ-CP
98/2024/NĐ-CP
52/2014/QH13
48/2010/QH12
192/2025/NĐ-CP
59/2020/QH14
71/2024/NĐ-CP
56/2024/TT-BTC
94/2024/NĐ-CP
96/2019/NĐ-CP
75/2025/NĐ-CP
57/2024/QH15
48/2024/TT-BTNMT
100/2024/NĐ-CP
123/2024/NĐ-CP
102/2024/NĐ-CP
61/2020/QH14
258/2025/NĐ-CP
40/2019/QH14
115/2024/NĐ-CP
29/2023/QH15
23/2025/TT-BNNMT
31/2024/QH15
27/2023/QH15
144/2025/NĐ-CP
